# Week 4 — Unit Test Writer

**AI-powered tool that automatically generates comprehensive unit tests for your code across multiple programming languages.** Paste your code, select your language (and optional test framework), and get a thorough test suite in seconds.


In [ ]:
# --- Copy from here into the Unit Test Writer ---

def is_valid_email(s: str) -> bool:
    """Return True if s looks like a valid email (has @ and a dot in the domain)."""
    if not s or "@" not in s:
        return False
    local, _, domain = s.partition("@")
    return bool(local.strip()) and "." in domain and len(domain) > 1


def slugify(text: str, max_length: int = 50) -> str:
    """Convert text to a URL-friendly slug: lowercase, spaces to hyphens, strip non-alphanumeric."""
    if not text:
        return ""
    slug = "".join(c if c.isalnum() or c.isspace() else "" for c in text)
    slug = "-".join(slug.lower().split()).strip("-")
    return slug[:max_length].rstrip("-") if len(slug) > max_length else slug


class BankAccount:
    """Simple account with deposit and withdraw; balance cannot go negative."""

    def __init__(self, balance: float = 0.0):
        if balance < 0:
            raise ValueError("Initial balance cannot be negative")
        self._balance = float(balance)

    @property
    def balance(self) -> float:
        return self._balance

    def deposit(self, amount: float) -> float:
        if amount < 0:
            raise ValueError("Deposit amount cannot be negative")
        self._balance += amount
        return self._balance

    def withdraw(self, amount: float) -> float:
        if amount < 0:
            raise ValueError("Withdraw amount cannot be negative")
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
        return self._balance

In [1]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

load_dotenv()

# OpenRouter client (OPENROUTER_API_KEY in .env)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

MODEL = "google/gemini-2.0-flash-001"  # fast and good for code

# Language -> default test framework
LANGUAGES = [
    "Python",
    "JavaScript",
    "TypeScript",
    "Java",
    "C#",
    "Go",
    "Ruby",
    "Rust",
]
FRAMEWORKS = {
    "Python": ["pytest", "unittest"],
    "JavaScript": ["Jest", "Mocha"],
    "TypeScript": ["Jest", "Vitest"],
    "Java": ["JUnit 5", "JUnit 4"],
    "C#": ["xUnit", "NUnit"],
    "Go": ["testing (stdlib)"],
    "Ruby": ["RSpec", "Minitest"],
    "Rust": ["cargo test"],
}

SYSTEM_PROMPT = """You are an expert software engineer who writes comprehensive unit tests. Given source code and the target language/framework, you produce a complete, runnable test suite.

Rules:
- Output only the test code. No markdown code fences, no explanation before or after.
- Cover normal cases, edge cases, and error cases where relevant.
- Use the standard testing framework for that language (as specified by the user).
- CRITICAL for Python: Make tests runnable without the user creating extra files. Either:
  (a) Put the code under test at the top of the same file (inline it), then write the tests below that import or use it directly (no "from your_module import ..."), OR
  (b) Use a single file with the implementation first, then "if __name__ == '__main__':" and the tests, so running the file runs pytest on the same file.
- Never use placeholder module names like "your_module" or "my_module". Use the real function/class names from the user's code and keep everything in one runnable file when possible.
- Include necessary imports and setup. If the user's code has multiple functions/classes, write tests for each."""

def get_framework_choices(lang):
    return FRAMEWORKS.get(lang, ["Standard"]) if lang else ["Standard"]

def generate_tests(code, language, framework):
    if not (code or "").strip():
        return "Paste your code above and select a language."
    lang = language or "Python"
    fw = framework or (FRAMEWORKS.get(lang, ["pytest"])[0] if lang == "Python" else "Standard")
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"""Language: {lang}
Test framework: {fw}

Generate a full unit test suite for this code. For Python: output one single file that includes the code under test (inlined at the top) and the tests, so it runs with "pytest file.py" or "python file.py" with no separate module. Do not use "your_module" or similar placeholders.

```
{code.strip()}
```"""},
            ],
            max_tokens=4096,
            extra_headers={"HTTP-Referer": "http://localhost:3000", "X-Title": "Unit Test Writer"},
        )
        text = (resp.choices[0].message.content or "").strip()
        if text.startswith("```"):
            lines = text.split("\n")
            if lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip() == "```":
                lines = lines[:-1]
            text = "\n".join(lines)
        return f"```\n{text}\n```"
    except Exception as e:
        return f"❌ Error: {e}"

# Gradio UI
with gr.Blocks(title="Unit Test Writer") as demo:
    gr.Markdown("### Paste your code, choose language and framework, then generate tests.")
    code_in = gr.Textbox(
        label="Your code",
        placeholder="Paste the code you want to test here...",
        lines=14,
    )
    with gr.Row():
        lang_drop = gr.Dropdown(label="Language", choices=LANGUAGES, value="Python")
        fw_drop = gr.Dropdown(label="Test framework", choices=get_framework_choices("Python"), value="pytest")
    def update_frameworks(lang):
        choices = get_framework_choices(lang)
        return gr.Dropdown(choices=choices, value=choices[0] if choices else "Standard")
    lang_drop.change(fn=update_frameworks, inputs=[lang_drop], outputs=[fw_drop])
    btn = gr.Button("Generate unit tests")
    out = gr.Markdown(label="Generated test suite")

    btn.click(
        fn=generate_tests,
        inputs=[code_in, lang_drop, fw_drop],
        outputs=out,
    )

demo.launch(inline=True)

C:\Users\kwx1494728\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
